In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib
import warnings
import threadpoolctl
from xgboost import XGBClassifier

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
N_CORES = 24
warnings.filterwarnings('ignore')

# PATHS - UPDATE THESE
MODEL_PATH = "/home/subhadeepd/AgeBow/NuQR_Plotting_3.12.25/Annotation_Model_Plots_29012026/model_output_HPT_Final_Holy/Final_XGB_Model_FullData.pkl"
ENCODER_PATH = "/home/subhadeepd/AgeBow/NuQR_Plotting_3.12.25/Annotation_Model_Plots_29012026/model_output_HPT_Final_Holy/final_label_encoder.pkl"
INPUT_ROOT_DIR = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP"
OUTPUT_ROOT_DIR = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated"

# ============================================================================
# 2. SETUP CPU AFFINITY
# ============================================================================
try:
    os.sched_setaffinity(0, set(range(N_CORES)))
except AttributeError:
    pass

threadpoolctl.threadpool_limits(limits=N_CORES)

# ============================================================================
# 3. PROCESSING FUNCTION
# ============================================================================
def process_single_file(file_path, output_path, model, le, expected_features):
    try:
        df = pd.read_csv(file_path)
        
        if df.empty:
            return

        # Feature Alignment
        missing_cols = [c for c in expected_features if c not in df.columns]
        if missing_cols:
            return 

        X_new = df[expected_features]

        # Prediction
        pred_indices = model.predict(X_new)
        pred_labels = le.inverse_transform(pred_indices)
        
        # Confidence
        probs = model.predict_proba(X_new)
        confidence_scores = np.max(probs, axis=1)

        # Saving
        df_out = df.copy()
        df_out['Predicted_Label'] = pred_labels
        df_out['Confidence_Score'] = confidence_scores

        df_out.to_csv(output_path, index=False)

    except Exception:
        pass # Silently ignore errors

# ============================================================================
# 4. MAIN BATCH LOOP
# ============================================================================
def main():
    # Load Model (Critical Check only)
    if not os.path.exists(MODEL_PATH) or not os.path.exists(ENCODER_PATH):
        sys.exit(1) # Exit silently with error code

    model = joblib.load(MODEL_PATH)
    le = joblib.load(ENCODER_PATH)
    
    try:
        expected_features = model.get_booster().feature_names
    except:
        sys.exit(1)

    # Walk Directory
    for root, dirs, files in os.walk(INPUT_ROOT_DIR):
        for file in files:
            if file.endswith(".csv"):
                
                full_input_path = os.path.join(root, file)
                
                # Mirror structure
                relative_path = os.path.relpath(full_input_path, INPUT_ROOT_DIR)
                full_output_path = os.path.join(OUTPUT_ROOT_DIR, relative_path)
                
                output_folder = os.path.dirname(full_output_path)
                os.makedirs(output_folder, exist_ok=True)
                
                process_single_file(full_input_path, full_output_path, model, le, expected_features)

if __name__ == "__main__":
    main()

In [4]:
#!/usr/bin/env python3

import os
import re
from glob import glob
from collections import defaultdict, Counter
import multiprocessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# STYLE
# =============================================================================

sns.set_theme(style="ticks")

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.spines.right": False,
    "axes.spines.top": False
})

# =============================================================================
# CONFIG
# =============================================================================

CELL_TYPE_COL = "Predicted_Label"

DATA_DIR = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated"

OUTPUT_CSV = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_celltype_counts_by_case.csv"
OUTPUT_PLOT = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_nested_pie_casewise.pdf"

NUM_CORES = 20
# =============================================================================
# CASE EXTRACTION (ROBUST)
# =============================================================================

def extract_case_id(filepath):
    """
    Extract case number from full path (folder or filename).
    Works for:
    - SKIN CASE 34 -IMG-5/...
    - SKIN CASE 34 -IMG-5_tile_XXXX.csv
    """
    match = re.search(r"CASE[\s\-_]*(\d+)", filepath, re.IGNORECASE)
    if match:
        return match.group(1)
    return None

# =============================================================================
# WORKER FUNCTION
# =============================================================================

def process_single_csv(csv_file):
    local_counts = Counter()

    try:
        case_id = extract_case_id(csv_file)

        if case_id is None:
            print(f"⚠️ No CASE ID found in: {csv_file}")
            return local_counts

        df = pd.read_csv(csv_file)

        if CELL_TYPE_COL not in df.columns:
            print(f"⚠️ Missing column '{CELL_TYPE_COL}' in: {csv_file}")
            return local_counts

        value_counts = df[CELL_TYPE_COL].value_counts()

        for cell_type, count in value_counts.items():
            local_counts[(case_id, cell_type)] += int(count)

    except Exception as e:
        print(f"⚠️ Error in {csv_file}: {e}")

    return local_counts

# =============================================================================
# PARALLEL COUNTING
# =============================================================================

def count_cells_parallel(data_dir):
    # 🔥 recursive search (VERY IMPORTANT)
    csv_files = glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

    print(f"\n📂 Found {len(csv_files)} CSV files")

    # Debug sample paths
    for f in csv_files[:5]:
        print("DEBUG FILE:", f)

    if len(csv_files) == 0:
        raise ValueError("❌ No CSV files found. Check DATA_DIR path.")

    final_counts = defaultdict(int)

    with multiprocessing.Pool(processes=NUM_CORES) as pool:
        results = pool.map(process_single_csv, csv_files)

    print("\n🔄 Aggregating results...")

    for res in results:
        for key, count in res.items():
            final_counts[key] += count

    return final_counts

# =============================================================================
# PLOTTING
# =============================================================================

def plot_nested_pie(df, output_file):

    case_totals = df.groupby("Case_ID")["Count"].sum()
    case_cell = df.groupby(["Case_ID", "Cell_Type"])["Count"].sum()

    inner_sizes = case_totals.values
    inner_labels = case_totals.index.tolist()

    outer_sizes = []
    outer_labels = []
    outer_colors = []

    cmap = plt.cm.tab20
    color_idx = 0

    for case in inner_labels:
        if case not in case_cell.index:
            continue

        cells = case_cell.loc[case]

        for cell, count in cells.items():
            outer_sizes.append(count)
            outer_labels.append(cell)
            outer_colors.append(cmap(color_idx % 20))

        color_idx += 1

    def autopct_abs(values):
        total = np.sum(values)
        def _fmt(pct):
            val = int(round(pct * total / 100))
            return f"{val}" if val > 0 else ""
        return _fmt

    fig, ax = plt.subplots(figsize=(10, 10))

    # Inner ring (Cases)
    ax.pie(
        inner_sizes,
        labels=inner_labels,
        radius=0.6,
        autopct=autopct_abs(inner_sizes),
        wedgeprops=dict(width=0.3, edgecolor="white"),
    )

    # Outer ring (Cell types)
    ax.pie(
        outer_sizes,
        labels=outer_labels,
        radius=1.0,
        autopct=autopct_abs(outer_sizes),
        pctdistance=0.85,
        labeldistance=1.05,
        colors=outer_colors,
        wedgeprops=dict(width=0.35, edgecolor="white"),
    )

    ax.set_title("Cell Counts by Case and Cell Type")
    plt.tight_layout()
    plt.savefig(output_file, dpi=150)
    plt.close()

    print(f"\n✅ Saved plot: {output_file}")

# =============================================================================
# MAIN
# =============================================================================

def main():

    counts = count_cells_parallel(DATA_DIR)

    rows = []
    for (case_id, cell_type), count in counts.items():
        rows.append({
            "Case_ID": str(case_id),
            "Cell_Type": str(cell_type),
            "Count": int(count)
        })

    df = pd.DataFrame(rows)

    # 🔍 DEBUG
    print("\nDEBUG SUMMARY:")
    print(df.head())
    print("Columns:", df.columns.tolist())
    print("Total rows:", len(df))

    if df.empty:
        raise ValueError("❌ No data extracted. Check filenames, column names, or folder structure.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Saved CSV: {OUTPUT_CSV}")

    plot_nested_pie(df, OUTPUT_PLOT)

    print("\n🎉 DONE!")

# =============================================================================

if __name__ == "__main__":
    main()


📂 Found 17241 CSV files
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_22500.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_20000.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_18750.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_21250.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_13750.csv

🔄 Aggregating results...

DEBUG SUMMARY:
  Case_ID       Cell_Type   Count
0      29  Arrector Pili   186209
1      29   Hair follicle   93668
2      29      F

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ Saved plot: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_nested_pie_casewise.pdf

🎉 DONE!


In [5]:
#!/usr/bin/env python3

import os
import re
from glob import glob
from collections import defaultdict, Counter
import multiprocessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# =============================================================================
# STYLE
# =============================================================================

sns.set_theme(style="ticks")

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.spines.right": False,
    "axes.spines.top": False
})

# =============================================================================
# CONFIG
# =============================================================================

CELL_TYPE_COL = "Predicted_Label"

DATA_DIR = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated"

OUTPUT_CSV = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_celltype_counts_by_case_new.csv"
OUTPUT_PLOT = "/home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_nested_pie_casewise_new.pdf"

NUM_CORES = 20

# =============================================================================
# FIXED CELL TYPE COLORS
# =============================================================================

CELL_TYPE_COLORS = {
    "Keratinocyte": "#1f77b4",
    "Fibroblast": "#ff7f0e",
    "Blood Vessel": "#d62728",
    "Hair follicle": "#9467bd",
    "Sebaceous Gland": "#8c564b",
    "Sweat Gland": "#e377c2",
    "Adipocyte": "#7f7f7f",
    "Arrector Pili": "#2ca02c",
}

# =============================================================================
# CASE EXTRACTION
# =============================================================================

def extract_case_id(filepath):
    match = re.search(r"CASE[\s\-_]*(\d+)", filepath, re.IGNORECASE)
    if match:
        return match.group(1)
    return None

# =============================================================================
# WORKER FUNCTION
# =============================================================================

def process_single_csv(csv_file):
    local_counts = Counter()

    try:
        case_id = extract_case_id(csv_file)

        if case_id is None:
            print(f"⚠️ No CASE ID in: {csv_file}")
            return local_counts

        df = pd.read_csv(csv_file)

        if CELL_TYPE_COL not in df.columns:
            print(f"⚠️ Missing column in: {csv_file}")
            return local_counts

        # Clean labels
        df[CELL_TYPE_COL] = df[CELL_TYPE_COL].astype(str).str.strip()

        value_counts = df[CELL_TYPE_COL].value_counts()

        for cell_type, count in value_counts.items():
            local_counts[(case_id, cell_type)] += int(count)

    except Exception as e:
        print(f"⚠️ Error in {csv_file}: {e}")

    return local_counts

# =============================================================================
# PARALLEL COUNTING
# =============================================================================

def count_cells_parallel(data_dir):
    csv_files = glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

    print(f"\n📂 Found {len(csv_files)} CSV files")

    if len(csv_files) == 0:
        raise ValueError("❌ No CSV files found. Check DATA_DIR.")

    for f in csv_files[:5]:
        print("DEBUG FILE:", f)

    final_counts = defaultdict(int)

    with multiprocessing.Pool(processes=NUM_CORES) as pool:
        results = pool.map(process_single_csv, csv_files)

    print("\n🔄 Aggregating...")

    for res in results:
        for key, count in res.items():
            final_counts[key] += count

    return final_counts

# =============================================================================
# PLOTTING
# =============================================================================

def plot_nested_pie(df, output_file):

    case_totals = df.groupby("Case_ID")["Count"].sum()
    case_cell = df.groupby(["Case_ID", "Cell_Type"])["Count"].sum()

    inner_sizes = case_totals.values
    inner_labels = case_totals.index.tolist()

    outer_sizes = []
    outer_labels = []
    outer_colors = []

    for case in inner_labels:
        if case not in case_cell.index:
            continue

        cells = case_cell.loc[case]

        for cell, count in cells.items():
            outer_sizes.append(count)
            outer_labels.append(cell)

            color = CELL_TYPE_COLORS.get(cell, "#cccccc")
            outer_colors.append(color)

    def autopct_abs(values):
        total = np.sum(values)
        def _fmt(pct):
            val = int(round(pct * total / 100))
            return f"{val}" if val > 0 else ""
        return _fmt

    fig, ax = plt.subplots(figsize=(10, 10))

    # INNER (cases)
    ax.pie(
        inner_sizes,
        labels=inner_labels,
        radius=0.6,
        autopct=autopct_abs(inner_sizes),
        wedgeprops=dict(width=0.3, edgecolor="white"),
    )

    # OUTER (cell types)
    ax.pie(
        outer_sizes,
        labels=outer_labels,
        radius=1.0,
        autopct=autopct_abs(outer_sizes),
        pctdistance=0.85,
        labeldistance=1.05,
        colors=outer_colors,
        wedgeprops=dict(width=0.35, edgecolor="white"),
    )

    # LEGEND
    legend_elements = [
        Patch(facecolor=color, label=cell)
        for cell, color in CELL_TYPE_COLORS.items()
    ]

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.3, 1),
        loc="upper left",
        fontsize=8,
        title="Cell Types"
    )

    ax.set_title("Cell Counts by Case and Cell Type")

    plt.tight_layout()
    plt.savefig(output_file, dpi=150)
    plt.close()

    print(f"\n✅ Saved plot: {output_file}")

# =============================================================================
# MAIN
# =============================================================================

def main():

    counts = count_cells_parallel(DATA_DIR)

    rows = []
    for (case_id, cell_type), count in counts.items():
        rows.append({
            "Case_ID": str(case_id),
            "Cell_Type": str(cell_type),
            "Count": int(count)
        })

    df = pd.DataFrame(rows)

    print("\nDEBUG SUMMARY:")
    print(df.head())
    print("Columns:", df.columns.tolist())
    print("Rows:", len(df))

    if df.empty:
        raise ValueError("❌ No data extracted. Check structure or column names.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Saved CSV: {OUTPUT_CSV}")

    plot_nested_pie(df, OUTPUT_PLOT)

    print("\n🎉 DONE!")

# =============================================================================

if __name__ == "__main__":
    main()


📂 Found 17241 CSV files
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_22500.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_20000.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_18750.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_21250.csv
DEBUG FILE: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/Rajiv_Skin_Features_HEIP_CellType_Annotated/SKIN CASE 29 -IMG-1/SKIN CASE 29 -IMG-1_tile_10000_13750.csv


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


🔄 Aggregating...

DEBUG SUMMARY:
  Case_ID      Cell_Type   Count
0      29  Arrector Pili  186209
1      29  Hair follicle   93668
2      29     Fibroblast   37223
3      29      Adipocyte   24923
4      29    Sweat Gland   14457
Columns: ['Case_ID', 'Cell_Type', 'Count']
Rows: 64

✅ Saved CSV: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_celltype_counts_by_case_new.csv


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ Saved plot: /home/subhadeepd/AgeBow/Rajiv_Skin_processing/RGCI_nested_pie_casewise_new.pdf

🎉 DONE!
